# 04 — Especialização LoRA

Esta etapa adapta o modelo local às respostas estruturadas e às recusas. Os casos de treino incluem perguntas respondíveis e perguntas cuja resposta não consta na base. O treino usa recuperação de trechos relevantes e acrescenta as fontes do gabarito quando necessário. O código completo está em [`alertavida/treinar_lora.py`](../alertavida/treinar_lora.py).

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath(".."))
import json, pandas as pd
from pathlib import Path
from IPython.display import display
from alertavida.nucleo import RAIZ, carregar_jsonl
from alertavida.treinar_lora import MAXLEN, main, alvo_json


In [2]:
casos = carregar_jsonl(RAIZ/'avaliacao'/'casos_treino_ampliado.jsonl')
resumo_treino = pd.DataFrame([
    {'Métrica':'Total de exemplos','Valor':len(casos)},
    {'Métrica':'Com resposta sustentada','Valor':sum(c['consta'] for c in casos)},
    {'Métrica':'Com recusa (não consta)','Valor':sum(not c['consta'] for c in casos)},
    {'Métrica':'Perguntas únicas','Valor':len({c['pergunta'] for c in casos})}
])
amostra_treino = pd.DataFrame([
    {'Pergunta':c['pergunta'],'Resposta do gabarito':c['gabarito'].get('proximo_passo'),'Tem suporte':c['consta']}
    for c in casos[:5]
])
display(resumo_treino)
display(amostra_treino)

,Métrica,Valor
0,Total de exemplos,81
1,Com resposta sustentada,56
2,Com recusa (não consta),25
3,Perguntas únicas,60


,Pergunta,Resposta do gabarito,Tem suporte
0,O motorista estava embriagado?,NaN,False
1,Resuma a ocorrência e diga o que confirmar com...,Acionar o SAMU (192) e orientar a não moviment...,True
2,Qual a previsão do tempo para esta região?,NaN,False
3,Resuma a ocorrência e diga o que confirmar com...,Acionar o SAMU (192) e orientar a não moviment...,True
4,Resuma a ocorrência e diga o que confirmar com...,"Confirmar com o operador se há vítimas, quanta...",True


### Configuração e supervisão

O alvo supervisionado é o JSON do gabarito. Para cada pergunta, o pipeline recupera até três trechos e acrescenta as fontes corretas do gabarito quando não apareceram na busca. O prompt não recebe rótulos do conjunto de avaliação.

In [3]:
from peft import LoraConfig
from alertavida.treinar_lora import MAXLEN

config_lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj']
)
config_treino = pd.DataFrame([
    {'Parâmetro':'Modelo base','Valor':'Qwen/Qwen2.5-1.5B-Instruct'},
    {'Parâmetro':'Método','Valor':'PEFT LoRA'},
    {'Parâmetro':'Rank / alpha','Valor':'16 / 32'},
    {'Parâmetro':'Dropout','Valor':0.05},
    {'Parâmetro':'Módulos ajustados','Valor':', '.join(config_lora.target_modules)},
    {'Parâmetro':'Épocas / learning rate','Valor':'3 / 2e-4'},
    {'Parâmetro':'Batch / acumulação','Valor':'1 / 8'},
    {'Parâmetro':'Comprimento máximo','Valor':MAXLEN}
])
display(config_treino)

,Parâmetro,Valor
0,Modelo base,Qwen/Qwen2.5-1.5B-Instruct
1,Método,PEFT LoRA
2,Rank / alpha,16 / 32
3,Dropout,0.05
4,Módulos ajustados,"k_proj, o_proj, q_proj, v_proj"
5,Épocas / learning rate,3 / 2e-4
6,Batch / acumulação,1 / 8
7,Comprimento máximo,1024


In [4]:
from inspect import getsource
print('Montagem do alvo supervisionado (JSON e recusas)\n')
print(getsource(alvo_json))
print('\nPipeline executável de treinamento\n')
print(getsource(main))

Montagem do alvo supervisionado (JSON e recusas)

def alvo_json(c: dict) -> str:
    g = c["gabarito"]
    if c["consta"]:
        d = {"tipo_provavel": g["tipo_provavel"], "prioridade_sugerida": g["prioridade_sugerida"],
             "informacoes_identificadas": list(c["evento"].keys()),
             "informacoes_ausentes": g["informacoes_ausentes"], "proximo_passo": g["proximo_passo"],
             "referencias": g["trechos"], "recusa": False, "motivo_recusa": None}
    else:
        d = {"tipo_provavel": None, "prioridade_sugerida": None, "informacoes_identificadas": [],
             "informacoes_ausentes": [], "proximo_passo": None, "referencias": [], "recusa": True,
             "motivo_recusa": "A base de conhecimento não contém informação suficiente para responder."}
    return json.dumps(d, ensure_ascii=False)


Pipeline executável de treinamento

def main(caminho_dados=None, caminho_saida=None, epocas=3, taxa_aprendizado=2e-4):
    tok = AutoTokenizer.from_pretrained(LLM_ID)
 

### Executar o treinamento

O treinamento registrado já foi executado. Para reproduzi-lo, altere `EXECUTAR_TREINO` para `True`. O novo adaptador será salvo em um diretório separado e não substituirá o adaptador usado pela aplicação.

In [5]:
EXECUTAR_TREINO = False
if EXECUTAR_TREINO:
    saida_treino = RAIZ/'modelos'/'lora_alertavida_retreinado'
    if (saida_treino/'adapter_model.safetensors').exists():
        raise FileExistsError(f'Já existe um adaptador em {saida_treino}; escolha outro diretório para preservar o resultado anterior.')
    main(
        caminho_dados=RAIZ/'avaliacao'/'casos_treino_ampliado.jsonl',
        caminho_saida=saida_treino,
        epocas=3,
        taxa_aprendizado=2e-4
    )
else:
    print('Treinamento não repetido. Abaixo estão os registros e a comparação da execução concluída.')

Treinamento não repetido. Abaixo estão os registros e a comparação da execução concluída.


In [6]:
registro = json.loads((RAIZ/'avaliacao'/'diagnosticos'/'registro_treinamento_lora.json').read_text(encoding='utf8'))
resumo_execucao = pd.DataFrame([
    {'Registro':'Passos de atualização','Valor':registro['passos_de_atualizacao']},
    {'Registro':'Duração (segundos)','Valor':registro['duracao_segundos']},
    {'Registro':'Último loss registrado','Valor':registro['ultimo_loss_registrado']},
    {'Registro':'Época do último log','Valor':registro['epoca_do_ultimo_log']},
    {'Registro':'Semente fixa','Valor':'Não'}
])
display(resumo_execucao)
print(registro['loss_observacao'])

,Registro,Valor
0,Passos de atualização,33
1,Duração (segundos),1409
2,Último loss registrado,0.1738
3,Época do último log,2.79
4,Semente fixa,Não


Último registro de log observado perto do fim do treino; não é apresentado como train_loss final.


In [7]:
base = pd.read_csv(RAIZ/'avaliacao'/'diagnosticos'/'placar_validacao_base.csv')
experimental = pd.read_csv(RAIZ/'avaliacao'/'diagnosticos'/'placar_validacao_experimental.csv')
comparacao = pd.concat([base.assign(adaptador='Original'), experimental.assign(adaptador='LoRA experimental')], ignore_index=True)
comparacao[['adaptador','acerto','alucinacao','json_valido','citacao_correta','recusa_correta','segundos','n_casos']]

,adaptador,acerto,alucinacao,json_valido,citacao_correta,recusa_correta,segundos,n_casos
0,Original,0.958,0.0,1.0,0.917,1.000,4.942,24
1,LoRA experimental,0.875,0.0,1.0,1.000,0.958,5.118,24


**Resultado da seleção:** o adaptador original teve acerto 0,958 na validação, acima de 0,875 do LoRA experimental, então ele permaneceu como padrão. A validação tem 24 casos sintéticos e não mede generalização para ocorrências reais. O registro de loss disponível é o último ponto do log perto do fim do treino; não é apresentado como curva nem como `train_loss` final.